In [7]:
import torch
import torch.nn as nn
import re
import torch.optim as optim
from rich.progress import track
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [8]:
# text = """the young boy walked down the long road that led to the old village at the end of the valley the road was dusty and dry and the sun was high in the sky the boy carried a small brown bag on his back inside the bag was a loaf of bread a bottle of water and a letter from his mother the letter was for the old man who lived alone at the edge of the village the old man had been a farmer for most of his life and knew every tree and stone in the valley the boy had never met the old man but his mother said he was kind and wise and always had a story to tell the boy walked past a wide green field where some cows were eating grass slowly near a wooden fence the fence was old and some parts had fallen down a black dog ran along the fence barking at the cows but the cows did not care and kept eating the boy stopped to watch the dog for a moment and then continued walking down the road the wind began to blow softly and the tall grass on both sides of the road moved gently like waves on a calm sea the boy looked up at the sky and saw a few white clouds moving slowly toward the mountains in the distance the mountains were tall and dark and always had snow on their peaks even in summer as the boy got closer to the village he could hear the sound of water a small river ran through the middle of the village and the people had built a stone bridge over it long ago children were playing near the river throwing stones into the water and laughing the boy crossed the bridge and looked down at the clear water below he could see small fish swimming near the bottom of the river moving in and out of the shadows the village had about twenty small houses most of them made of stone with red or brown roofs a few old trees grew between the houses giving shade to the narrow streets some women were sitting outside their doors talking quietly while their children played nearby an old man was fixing a wooden cart near the well in the center of the village the boy asked him where the farmer lived and the old man pointed to a small house at the far end of the village near a large oak tree the boy thanked him and walked toward the house when he arrived he knocked on the wooden door softly after a moment the door opened and a tall old man with white hair and kind eyes looked down at him the boy held out the letter and said it was from his mother the old man smiled and took the letter and invited the boy inside the house was small but warm a fire was burning in the corner and the smell of soup filled the room the old man read the letter slowly by the light of the fire and then folded it carefully and put it in his coat pocket he thanked the boy and offered him a bowl of soup and a place to rest the boy sat down near the fire and the old man began to tell him a story about the valley long ago"""

import urllib.request
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "text.txt")
with open("text.txt") as f:
    text = f.read(80000)  # first 80k chars

data = text.lower().split('\n')

# text = re.sub(r'[^a-zA-Z\s]','',text)

words = text.lower().split()

vocab = sorted(set(words))
wti   = {w:i for i,w in enumerate(vocab)}
itw   = {i:w for i,w in enumerate(vocab)}

seq = 25

X = []
y = []

for i in range(len(words)-seq):
  X.append([wti[w] for w in words[i:i+seq]])
  y.append([wti[words[i+seq]]])

X = torch.tensor(X)
y = torch.tensor(y)

In [9]:
class MyLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.fh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.fx = nn.Linear(input_size,  hidden_size, bias=False)
        self.fb = nn.Parameter(torch.randn(hidden_size, device=device))

        self.ih = nn.Linear(hidden_size, hidden_size, bias=False)
        self.ix = nn.Linear(input_size,  hidden_size, bias=False)
        self.ib = nn.Parameter(torch.randn(hidden_size, device=device))

        self.gh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.gx = nn.Linear(input_size,  hidden_size, bias=False)
        self.gb = nn.Parameter(torch.randn(hidden_size, device=device))

        self.oh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.ox = nn.Linear(input_size,  hidden_size, bias=False)
        self.ob = nn.Parameter(torch.randn(hidden_size, device=device))

        #r1 adding confidence layer (Gradient Boost Gate)
        # self.confidence_layer = nn.Linear(input_size+hidden_size+hidden_size, 1)

        #r2 boost amplification
        # self.boost_factor = nn.Linear(hidden_size,hidden_size)

    def compute_forgot(self,h,x):
        return torch.sigmoid(self.fh(h) + self.fx(x) + self.fb)

    def compute_input(self,h,x):
        return torch.sigmoid(self.ih(h) + self.ix(x) + self.ib)

    def compute_gate(self,h,x):
        return torch.tanh(self.gh(h) + self.gx(x) + self.gb)

    def compute_output(self,h,x):
        return torch.sigmoid(self.oh(h) + self.ox(x) + self.ob)

    def lstm_cell(self,x,h_prev, c_prev):
        f = self.compute_forgot(h_prev,x)
        i = self.compute_input(h_prev, x)
        g = self.compute_gate(h_prev,x)
        o = self.compute_output(h_prev, x)

        cell_state   = (c_prev * f)+(i * g)
        hidden_state = o * torch.tanh(cell_state)

        return hidden_state, cell_state

    def forward(self, x, h0=None, c0=None):
        seq_len, batch, _ = x.shape
        h = h0 if h0 is not None else torch.zeros(batch, self.hidden_size, device=x.device)
        c = c0 if c0 is not None else torch.zeros(batch, self.hidden_size, device=x.device)



        outputs = []
        for t in range(seq_len):
            h, c = self.lstm_cell(x[t], h, c)

            # print(f"x shape: {x.shape}")
            # print(f"h shape: {h.shape}")
            # print(f"c shape: {c.shape}")
            # combined = torch.cat([x[t], h, c],dim=-1)

            # confidence = torch.sigmoid(self.confidence_layer(combined))

            # boost = self.boost_factor(h)
            # boost = torch.tanh(self.boost_factor(h))


            # h = h + (1 - confidence) * boost
            # h = h + confidence * boost

            outputs.append(h.unsqueeze(0))
        return torch.cat(outputs, dim=0)


# embedding = nn.Embedding(len(words), 100).to(device)

model = MyLSTM(100, 100).to(device)

embedding = nn.Embedding(len(vocab), 100).to(device)
ann_layer = nn.Linear(100, len(vocab)).to(device)

# ann_layer = nn.Linear(100, len(words)).to(device)

In [10]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(model.parameters())+list(ann_layer.parameters()), lr=0.01)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-5)

In [11]:
data_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X,y), batch_size=10, drop_last=True,shuffle=True)

In [13]:
for epoch in track(range(50), description="Training Data: "):
  epoch_loss = None
  for X_train, y_true in data_loader:
    X_train, y_true = X_train.to(device), y_true.to(device)

    # In PyTorch, .to() on tensors is NOT in-place.
    # It must be assigned: embeddings = embeddings.to(device)
    embeddings = embedding(X_train).to(device)

    # Permute to (seq_len, batch, feature) for MyLSTM
    output = model.forward(embeddings.permute(1, 0, 2))

    y_pred = ann_layer(output[-1])
    loss = loss_fn(y_pred, y_true.squeeze())
    epoch_loss = loss.item()
    optimizer.zero_grad()
    loss.backward()

    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
  scheduler.step()
  print(f"loss: {epoch_loss}")

Output()

loss: 4.467820644378662

loss: 4.831841468811035

loss: 3.870466709136963

loss: 3.7108941078186035

...

loss: 0.6146740317344666

loss: 0.9768482446670532

loss: 0.5994168519973755

loss: 0.1070525050163269

loss: 0.27202701568603516

In [14]:
import pickle as pkl

with open('model-default-boost.pkl','wb') as file:
  pkl.dump(model,file)

In [16]:
def generate_response(model, text, max_tokens=20, temperature=0.8, top_k=0, top_p=0.75):
  current_input_words = text.split()
  final_data_words = list(current_input_words)

  print(text)
  model.eval()

  with torch.no_grad():
    for i in range(max_tokens):
      model_input_words = current_input_words[-seq:]
      data = [wti[chunk.lower().strip()] for chunk in model_input_words]
      data = torch.tensor([data]).to(device)

      embeddings = embedding(data)
      output = model.forward(embeddings.permute(1, 0, 2))

      output = ann_layer(output[-1])

      probabilities = F.softmax(output / temperature, dim=-1)

      if top_p < 1.0:
          sorted_probs, sorted_indices = torch.sort(probabilities, descending=True)
          cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
          num_to_keep = (cumulative_probs < top_p).sum(dim=-1) + 1
          mask = torch.arange(probabilities.shape[-1], device=device).unsqueeze(0) < num_to_keep.unsqueeze(1)
          filtered_sorted_probs = sorted_probs * mask
          probabilities = torch.zeros_like(probabilities).scatter_(-1, sorted_indices, filtered_sorted_probs)
          probabilities = probabilities / probabilities.sum(dim=-1, keepdim=True)

      word_idx = torch.multinomial(probabilities, 1).item()
      predicted_word = itw[word_idx]

      final_data_words.append(predicted_word)
      current_input_words.append(predicted_word)

  return ' '.join(final_data_words)

generate_response(model, "the", max_tokens=1000, temperature=1, top_k=0, top_p=0.75)

the


"the people are abused; set on. this paltering becomes not rome, nor has coriolanus deserved this so dishonour'd rub, laid falsely i' the plain way of his merit. coriolanus: tell cominius: should we were chosen what is rather to be not to be their bedfellow. worthy cominius, speak. nay, keep your place. first senator: sit, coriolanus; never shame to hear what you have nobly done. coriolanus: your horror's pardon: i think, you'll with ever made me, have not been as fair as sound as when he did need your loves, and do you think that his contempt shall not be bruising to you, when he did contemn what he requested should be in them to give. brutus: come, we'll inform them of our proceedings here: on the marketplace, i know, they do attend us. first citizen: once, if he do require our voices, we ought not to deny him. second citizen: we may, sir, if we will. third citizen: we have power his wounds and tell you think to fob off our disgrace with a tale: but, an 't please you, deliver. meneni